# Create a journal analysis project

Run all cells, then choose **Random journal sample** or **Manual DOI selection**.

For a random sample, search for a journal, select it, choose dates and the number of papers, then create the project. For manual selection, search one DOI at a time (press **Enter** to look it up, review the dropdown and **Add paper**), or check a newline-separated DOI list. Full doi.org URLs and bare DOIs both work. Review the selected papers, author names, journal names, paper count and year range before creating the project. A warning identifies selections spanning multiple journals.

Manual lookup preloads the complete metadata needed for the exports. Project creation reuses it without further API requests. Existing projects are never overwritten. PDFs are not downloaded; choose your own folders for PDFs and analysis results.

Dependencies and setup instructions are in `requirements.txt` and `dev/SETUP.md`. Keep this notebook with the `app/` folder and `clean_and_count_text.ipynb`.

Editorials, front matter, back matter and similar non-scientific items are excluded in both selection modes. Review the recorded reasons in the selection report.


In [ ]:
# Uncomment if notebook dependencies are missing, then restart the kernel:
# %pip install -r requirements.txt

In [ ]:
from datetime import datetime
import os
from html import escape
from pathlib import Path
from zoneinfo import ZoneInfo

import ipywidgets as widgets
from IPython.display import display
from app import journal_tools as tools

# Run All Cells may reuse an import from before the tools were updated.
import importlib
if getattr(tools, "PROJECT_NOTEBOOK_API_VERSION", None) != 3:
    tools = importlib.reload(tools)
if getattr(tools, "PROJECT_NOTEBOOK_API_VERSION", None) != 3:
    raise RuntimeError("The project notebook and app/journal_tools.py do not match. Update the complete tools folder, reload the notebook page, and run all cells again.")

TOOLS_ROOT = Path(tools.__file__).resolve().parent.parent
TODAY = datetime.now(ZoneInfo("Europe/Zurich")).date()
DEFAULT_PROJECTS_ROOT = Path(os.environ.get("JOURNAL_TOOLS_PROJECTS_ROOT", TOOLS_ROOT.parent))


In [ ]:
field_style = {"description_width": "150px"}
wide = widgets.Layout(width="100%")
journal_query = widgets.Text(description="Journal title / ISSN:", placeholder="Enter a journal title or ISSN", layout=wide, style=field_style)
email_field = widgets.Text(description="Contact email:", placeholder="Optional, used only for Crossref requests", layout=wide, style=field_style)
search_button = widgets.Button(description="Find journal", button_style="info")
journal_choice = widgets.Dropdown(description="Select journal:", options=[], value=None, layout=wide, style=field_style)
start_field = widgets.DatePicker(description="Start date:", value=tools.months_before(TODAY, 12), style=field_style)
end_field = widgets.DatePicker(description="End date:", value=TODAY, style=field_style)
count_field = widgets.BoundedIntText(description="Random papers:", value=10, min=1, max=1000000, style=field_style)
basis_field = widgets.Dropdown(description="Publication date:", options=[("First known publication", "first"), ("Online publication", "online"), ("Print publication", "print")], value="first", style=field_style, layout=wide)
root_field = widgets.Text(description="Projects parent:", value=str(DEFAULT_PROJECTS_ROOT), style=field_style, layout=wide)
folder_field = widgets.Text(description="Project folder:", placeholder="Optional: leave empty for an automatic date + journal + range name", style=field_style, layout=wide)
seed_field = widgets.Text(description="Random seed:", placeholder="Optional integer; empty = a fresh recorded seed", style=field_style, layout=wide)
archive_field = widgets.Text(description="Archived JSON:", placeholder="Optional path to a saved *_crossref.json for an offline replay", style=field_style, layout=wide)
create_button = widgets.Button(description="Create project", button_style="primary", disabled=True, layout=widgets.Layout(width="180px"))
preview = widgets.HTML()
search_output = widgets.Output()
creation_output = widgets.Output()
last_project = None


def update_preview(change=None):
    if project_mode.value == "manual":
        update_manual_preview()
        return
    journal = journal_choice.value
    create_button.disabled = journal is None
    if journal is None:
        preview.value = "<p>Find and select a journal to preview the project folder.</p>"
        return
    try:
        tools.validate_window(start_field.value, end_field.value, basis_field.value)
        name = folder_field.value.strip() or tools.project_folder_name(journal, start_field.value, end_field.value)
        path = Path(root_field.value).expanduser() / name
        preview.value = f"<p><strong>New project:</strong> {escape(str(path))}</p>"
    except ValueError as error:
        preview.value = f"<p>{escape(str(error))}</p>"
        create_button.disabled = True


def search_journal(button=None):
    search_button.disabled = True
    journal_choice.options = []
    search_output.clear_output()
    try:
        with search_output:
            print("Looking up journals…")
            candidates = tools.find_journals(journal_query.value, email_field.value.strip() or None)
            if not candidates:
                print("No journals found. Try a more specific title or an ISSN.")
            else:
                journal_choice.options = [("Choose the intended journal…", None)] + [
                    (f"{journal.name} ({', '.join(journal.issns)})", journal) for journal in candidates
                ]
                print(f"Found {len(candidates)} candidate(s). Select the intended journal below.")
    except Exception as error:
        with search_output:
            print(f"Journal lookup failed: {error}")
    finally:
        search_button.disabled = False
        update_preview()


def query_changed(change):
    # A changed search must not reuse a journal selected for the previous query.
    journal_choice.options = []
    search_output.clear_output()
    update_preview()


def make_project(button=None):
    global last_project
    create_button.disabled = True
    search_button.disabled = True
    creation_output.clear_output()
    try:
        with creation_output:
            if project_mode.value == "manual":
                make_manual_project()
                return
            if journal_choice.value is None:
                raise ValueError("Find and select a journal first.")
            if not root_field.value.strip():
                raise ValueError("Enter the parent folder for your projects.")
            seed = int(seed_field.value.strip()) if seed_field.value.strip() else None
            print("Fetching metadata and preparing the project. This can take a few minutes…")
            result = tools.create_project(
                journal_choice.value, start_field.value, end_field.value,
                count_field.value, root_field.value.strip(),
                date_basis=basis_field.value, seed=seed,
                email=email_field.value.strip() or None,
                project_name=folder_field.value.strip() or None,
                input_json=archive_field.value.strip() or None,
            )
            last_project = result
            print(f"Created: {result['path']}")
            print(f"Full export: {len(result['rows'])} records; eligible: {result['selection_report']['eligible_entry_count']}")
            print(f"Selected papers: {len(result['selected'])}; saved seed: {result['config']['seed']}")
            for index, row in enumerate(result['selected'], 1):
                print(f"{index}. {row['title']} ({row['doi']})")
            print("Open the new project's clean_and_count_text.ipynb. Organize PDFs and analysis results as you prefer.")
    except Exception as error:
        with creation_output:
            print(f"Project was not created: {error}")
    finally:
        search_button.disabled = False
        update_preview()


# Manual selection shares project destination settings but never uses sampling controls.
project_mode = widgets.ToggleButtons(description="Selection:", options=[("Random journal sample", "random"), ("Manual DOI selection", "manual")], value="random", style=field_style)
manual_state = {"cache": {}, "selected": {}, "busy": False, "bulk_checked": None, "bulk_errors": []}
manual_doi_query = widgets.Text(description="Paper DOI:", placeholder="Paste a DOI or https://doi.org/... and press Enter", style=field_style, layout=wide)
manual_search_button = widgets.Button(description="Search DOI", button_style="info")
manual_search_status = widgets.HTML()
manual_result_dropdown = widgets.Accordion(children=[], selected_index=None, layout=widgets.Layout(width="100%", display="none"))
manual_bulk_input = widgets.Textarea(description="DOI list:", placeholder="One DOI or doi.org URL per line", style=field_style, layout=widgets.Layout(width="100%", height="160px"))
manual_bulk_button = widgets.Button(description="Check and add DOI list", button_style="info", layout=widgets.Layout(width="210px"))
manual_bulk_status = widgets.HTML()
manual_selected_rows = widgets.VBox(layout=widgets.Layout(width="100%"))
manual_overview_html = widgets.HTML()
manual_overview_html.add_class("manual-project-overview")


def manual_paper_html(record):
    row = tools.manual_record_row(record)
    authors = "; ".join(
        " ".join(filter(None, (tools.plain_text(author.get("given")), tools.plain_text(author.get("family")))))
        or tools.plain_text(author.get("name")) for author in record.get("author", [])
    ) or "Not available"
    return (
        '<div style="padding:8px;overflow-wrap:anywhere;min-width:0;line-height:1.5;">'
        f'<strong style="display:block;font-size:1.05em;">{escape(row["title"])}</strong>'
        f'<div><b>Authors:</b> {escape(authors)}</div>'
        f'<div><b>Journal:</b> {escape(row["journal"])}</div>'
        f'<div><b>Year:</b> {escape(row["year"])}</div>'
        f'<div><b>DOI:</b> {escape(row["doi"])}</div></div>')


def manual_bulk_pending():
    return bool(manual_bulk_input.value.strip() and
                (manual_state["bulk_checked"] != manual_bulk_input.value or manual_state["bulk_errors"]))


def update_manual_preview():
    overview = tools.manual_overview(manual_state["selected"].values())
    count = overview["paper_count"]
    create_button.disabled = not count or manual_state["busy"] or manual_bulk_pending()
    if not count:
        manual_overview_html.value = "<p>No papers selected. Search for a DOI or check a DOI list.</p>"
        preview.value = "<p>Select at least one checked paper to preview the project folder.</p>"
        return
    journals = "; ".join(f"{item['name']} ({item['paper_count']})" for item in overview["journals"])
    warning = ('<p role="alert" style="color:#8a5700;"><strong>Warning:</strong> Papers from multiple journals are selected.</p>'
               if overview["multiple_journals"] else "")
    manual_overview_html.value = (
        f'<p><strong>Journals:</strong> {escape(journals)}<br>'
        f'<strong>Papers:</strong> {count}<br><strong>Year range:</strong> {overview["year_start"]}–{overview["year_end"]}</p>{warning}')
    try:
        name = folder_field.value.strip() or tools.manual_project_folder_name(overview)
        path = Path(root_field.value).expanduser() / name
        pending = "<br>Check the current DOI list before creating the project." if manual_bulk_pending() else ""
        preview.value = f"<p><strong>New project:</strong> {escape(str(path))}{pending}</p>"
    except ValueError as error:
        preview.value = f"<p>{escape(str(error))}</p>"
        create_button.disabled = True


def manual_refresh_selected():
    rows = []
    for doi, record in manual_state["selected"].items():
        remove = widgets.Button(description="Remove", tooltip="Remove " + doi, layout=widgets.Layout(width="95px", flex="0 0 95px"))
        remove.on_click(lambda _, key=doi: manual_remove_paper(key))
        row = widgets.HBox([widgets.HTML(manual_paper_html(record), layout=widgets.Layout(width="100%", min_width="0")), remove],
                           layout=widgets.Layout(width="100%", align_items="center", border="1px solid #d9dfe6", margin="4px 0"))
        row.add_class("manual-selected-paper")
        rows.append(row)
    manual_selected_rows.children = tuple(rows)
    update_preview()


def manual_remove_paper(doi):
    manual_state["selected"].pop(doi, None)
    manual_refresh_selected()


def manual_add_paper(record):
    doi = tools.normalize_doi(record["DOI"])
    exists = doi in manual_state["selected"]
    manual_state["selected"].setdefault(doi, record)
    manual_doi_query.value = ""
    manual_result_dropdown.children = ()
    manual_result_dropdown.layout.display = "none"
    manual_search_status.value = "<p>Paper already selected.</p>" if exists else "<p>Paper added to the selection below.</p>"
    manual_refresh_selected()


def manual_query_changed(change=None):
    manual_result_dropdown.children = ()
    manual_result_dropdown.layout.display = "none"
    manual_search_status.value = ""


def manual_finish_lookup(status_widget):
    """Always release controls, including when preparing or refreshing the form fails."""
    manual_state["busy"] = False
    manual_search_button.disabled = manual_bulk_button.disabled = False
    try:
        update_preview()
    except Exception as error:
        create_button.disabled = True
        status_widget.value = f"<p role='alert'>Could not update the project form: {escape(str(error))}</p>"


def manual_search_doi(button=None):
    if manual_state["busy"]:
        return
    manual_state["busy"] = True
    try:
        manual_search_button.disabled = manual_bulk_button.disabled = True
        manual_query_changed()
        manual_search_status.value = "<p>Looking up the DOI and preloading paper metadata…</p>"
        update_preview()
        record = tools.lookup_doi(manual_doi_query.value, manual_state["cache"], email_field.value.strip() or None)
        choose = widgets.Button(description="Add paper", button_style="success", layout=widgets.Layout(width="150px", height="32px", min_height="32px", flex="0 0 32px"))
        choose.on_click(lambda _: manual_add_paper(record))
        result = widgets.VBox([widgets.HTML(manual_paper_html(record), layout=widgets.Layout(width="100%", min_width="0", flex="0 0 auto")), choose],
                              layout=widgets.Layout(width="100%", max_height="320px", overflow="hidden auto"))
        result.add_class("manual-doi-result")
        manual_result_dropdown.children = (result,)
        manual_result_dropdown.set_title(0, "Select found paper")
        manual_result_dropdown.layout.display = ""
        manual_result_dropdown.selected_index = 0
        manual_search_status.value = "<p>Review the result in the dropdown, then choose Add paper.</p>"
    except Exception as error:
        manual_search_status.value = f"<p role='alert'>DOI lookup failed: {escape(str(error))}</p>"
    finally:
        manual_finish_lookup(manual_search_status)


def manual_bulk_changed(change=None):
    manual_state["bulk_checked"] = None
    manual_state["bulk_errors"] = []
    manual_bulk_status.value = "<p>Press Check and add DOI list to validate these entries.</p>" if manual_bulk_input.value.strip() else ""
    update_preview()


def manual_check_bulk(button=None):
    if manual_state["busy"]:
        return
    manual_state["busy"] = True
    try:
        manual_search_button.disabled = manual_bulk_button.disabled = True
        manual_bulk_status.value = "<p>Checking all DOIs and preloading metadata…</p>"
        update_preview()
        if not manual_bulk_input.value.strip():
            raise ValueError("Paste at least one DOI, with one entry per line.")
        result = tools.check_dois(manual_bulk_input.value, manual_state["cache"], email_field.value.strip() or None)
        manual_state["bulk_checked"] = manual_bulk_input.value
        manual_state["bulk_errors"] = result["errors"]
        if result["errors"]:
            entries = "".join(f"<li>Line {item['line']}: {escape(item['input'])} — {escape(item['error'])}</li>" for item in result["errors"])
            manual_bulk_status.value = ('<p role="alert">No papers from this list were added. Correct these entries and check again; '
                                       'successful lookups are retained for retry.</p><ul>' + entries + '</ul>')
        else:
            before = len(manual_state["selected"])
            for record in result["records"]:
                manual_state["selected"].setdefault(tools.normalize_doi(record["DOI"]), record)
            added = len(manual_state["selected"]) - before
            manual_bulk_status.value = f"<p>Checked {len(result['records'])} unique DOI(s); added {added} paper(s). Repeated or already selected DOIs are included once.</p>"
        manual_refresh_selected()
    except Exception as error:
        manual_bulk_status.value = f"<p role='alert'>Could not check DOI list: {escape(str(error))}</p>"
    finally:
        manual_finish_lookup(manual_bulk_status)


def make_manual_project():
    global last_project
    if manual_bulk_pending():
        raise ValueError("Check and correct the current DOI list, or clear it before proceeding.")
    if not root_field.value.strip():
        raise ValueError("Enter the parent folder for your projects.")
    result = tools.create_manual_project(list(manual_state["selected"].values()), root_field.value.strip(),
                                         project_name=folder_field.value.strip() or None)
    last_project = result
    print(f"Created: {result['path']}")
    print(f"Manually selected papers: {len(result['selected'])}. Metadata reused from DOI lookup; no random sampling applied.")
    for index, row in enumerate(result["selected"], 1):
        print(f"{index}. {row['title']} — {row['journal']} ({row['year']}) — {row['doi']}")
    print("Open the new project's clean_and_count_text.ipynb to categorize its selected papers.")


search_button.on_click(search_journal)
create_button.on_click(make_project)
journal_query.observe(query_changed, names="value")
for field in (journal_choice, start_field, end_field, basis_field, root_field, folder_field):
    field.observe(update_preview, names="value")
update_preview()
advanced = widgets.Accordion(children=[widgets.VBox([basis_field, seed_field, archive_field])], selected_index=None)
advanced.set_title(0, "Publication date, reproducibility and offline replay")
random_panel = widgets.VBox([journal_query, search_button, search_output, journal_choice,
                            widgets.HBox([start_field, end_field]), count_field, advanced])
manual_entry_tabs = widgets.Tab(children=[
    widgets.VBox([manual_doi_query, manual_search_button, manual_search_status, manual_result_dropdown]),
    widgets.VBox([manual_bulk_input, manual_bulk_button, manual_bulk_status]),
])
manual_entry_tabs.set_title(0, "Search one DOI")
manual_entry_tabs.set_title(1, "Paste DOI list")
manual_panel = widgets.VBox([
    widgets.HTML("<p>Select journal articles by DOI. Full doi.org URLs and bare DOIs both work. Metadata is preloaded during lookup.</p>"),
    manual_entry_tabs, widgets.HTML("<h4>Selected papers</h4>"), manual_selected_rows,
    widgets.HTML("<h4>Selection overview</h4>"), manual_overview_html,
], layout=widgets.Layout(width="100%", display="none"))


def update_project_mode(change=None):
    manual = project_mode.value == "manual"
    random_panel.layout.display = "none" if manual else ""
    manual_panel.layout.display = "" if manual else "none"
    update_preview()


manual_search_button.on_click(manual_search_doi)
# Text's submit event distinguishes Enter from merely leaving the input field.
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    manual_doi_query.on_submit(manual_search_doi)
manual_doi_query.observe(manual_query_changed, names="value")
manual_bulk_input.observe(manual_bulk_changed, names="value")
manual_bulk_button.on_click(manual_check_bulk)
project_mode.observe(update_project_mode, names="value")
update_project_mode()
display(widgets.VBox([
    widgets.HTML("<p>Editorials, front/back matter and similar non-scientific articles are excluded in both selection modes.</p>"),
    project_mode, email_field, random_panel, manual_panel,
    root_field, folder_field, preview, create_button, creation_output,
]))


## Generated structure

```text
<creation-date>_journalanalysis_<journal>_<start>_<end>/
├── project.json
├── README.md
├── clean_and_count_text.ipynb
└── output/
    ├── <journal>_<start>_<end>.csv
    ├── <journal>_<start>_<end>.bib
    ├── <journal>_<start>_<end>_crossref.json
    ├── <journal>_<start>_<end>_report.json
    ├── <journal>_<start>_<end>_random<N>.csv
    ├── <journal>_<start>_<end>_random<N>.bib
    └── <journal>_<start>_<end>_random<N>_selection.json
```

The default parent is the tools folder's parent, so new projects sit alongside the tools folder. You can change it in the form. Folder dates use Europe/Zurich. No PDFs are downloaded. Choose your own folders and files for PDFs and analysis results.

The default date basis is the **earliest known publication date** across online, print, published and issued metadata. Dates are inclusive. Partial dates use the first day for filtering while exports keep their original precision. CSV and BibTeX contain the same DOIs and citation keys in the same order.

The sample is uniform without replacement across eligible DOIs. Editorials, front matter, back matter and similar non-scientific items are excluded using publisher article-type metadata where available and title rules. All exclusions and the actual seed are recorded. Review those exclusions and compare the text-cleaning results with the source PDF. Crossref coverage depends on publisher deposits. [Crossref REST API documentation](https://www.crossref.org/documentation/retrieve-metadata/rest-api/).

Manual projects use `_manual` in their export names and `_selected<N>` for the chosen-paper CSV/BibTeX and selection report. Manual DOI checks reject editorials, front matter, back matter and similar non-scientific items as well. Their order follows selection order; duplicate DOIs are included once. The `sample_export` project field also points to this selected-paper CSV, so the categorization notebook lists these papers automatically. Multi-journal projects record all journal names and identifiers and retain the warning in their README and reports. Year range uses the displayed bibliography year (print year when available); project date bounds use first known publication dates.


## Optional: create a project from code

The form uses the same function as the example below. For an offline replay, construct `tools.Journal("Journal title", ("1234-5679",))` with the real journal identifiers and pass a path as `input_json`. This also avoids a journal-lookup request. Keep the creation call commented until you have chosen your settings.

In [ ]:
# from datetime import date
# candidates = tools.find_journals("Your journal title or ISSN")
# for index, candidate in enumerate(candidates):
#     print(index, candidate)
# journal = candidates[0]  # Select the intended journal after reviewing the list.
# result = tools.create_project(
#     journal=journal,
#     start=date(2025, 11, 4),
#     end=date(2026, 11, 4),
#     count=10,
#     projects_root=DEFAULT_PROJECTS_ROOT,
#     seed=42,  # Optional: omit for a fresh recorded seed.
#     # input_json=Path("path/to/saved_crossref.json"),
# )

## Optional: draw another sample without fetching again

This writes a separate sample alongside the archived source and preserves the first sample. To create another complete analysis project, replay the archived JSON into a new project folder instead.

In [ ]:
# import csv
# import json
# if last_project is None:
#     raise ValueError("Create a project first, or set project_path to an existing project.")
# project_path = last_project["path"]
# config = json.loads((project_path / "project.json").read_text(encoding="utf-8"))
# source = project_path / config["full_export"]
# with source.open(encoding="utf-8-sig", newline="") as handle:
#     population = list(csv.DictReader(handle))
# selected, report = tools.sample_rows(population, count=10)  # Optional seed=42.
# stem = f"{source.stem}_random{len(selected)}_seed{report['seed']}"
# if any((source.parent / (stem + suffix)).exists() for suffix in (".csv", ".bib", "_selection.json")):
#     raise FileExistsError("This sample already exists; choose another seed.")
# tools.write_selection(selected, source.parent, stem, dict(report, source_csv=config["full_export"]))
# print(source.parent / (stem + ".csv"))